#### Cardiovascular Death Rate by Age Group — Preprocessing

In [11]:
import pandas as pd
import numpy as np
import pycountry
import warnings
warnings.filterwarnings('ignore')

#### 1. Load Raw Data

In [12]:
RAW_PATH = '../data/raw/cardiovascular-disease-death-rate-age-group-who-mdb.csv'

df = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")
print(f"Unique entities: {df['Entity'].nunique()}")
df.head(3)

Raw shape: (4904, 22)
Columns: ['Entity', 'Code', 'Year', 'Under 1s', '1-4 years', '5-9 years', '10-14 years', '15-19 years', '20-24 years', '25-29 years', '30-34 years', '35-39 years', '40-44 years', '45-49 years', '50-54 years', '55-59 years', '60-64 years', '65-69 years', '70-74 years', '75-79 years', '80-84 years', 'Over 85s']
Year range: 1950 – 2023
Unique entities: 117


,Entity,Code,Year,Under 1s,1-4 years,5-9 years,10-14 years,15-19 years,20-24 years,25-29 years,...,40-44 years,45-49 years,50-54 years,55-59 years,60-64 years,65-69 years,70-74 years,75-79 years,80-84 years,Over 85s
0,Albania,ALB,1987,6.203474,13.636364,1.801261,1.898133,6.443299,6.160164,12.508687,...,36.376606,71.830986,180.38237,346.77420,621.44700,1063.3694,2375.0000,3824.6753,5753.0120,12110.000
1,Albania,ALB,1988,7.299270,13.369901,5.002943,2.481390,6.946637,6.709158,7.833787,...,37.037037,76.657455,149.95924,303.65973,636.70886,1140.9090,2331.7307,4206.3490,5431.9526,12126.214
2,Albania,ALB,1989,1.200480,12.842176,2.611724,1.834862,5.605730,5.954350,8.733624,...,38.620690,62.627636,155.82329,301.46340,620.47440,1192.8251,2293.8389,4514.1064,6313.9536,11115.385


#### 2. Filter Years ≥ 2000

In [13]:
df = df[df['Year'] >= 2000].copy()
print(f"After year filter (≥ 2000): {df.shape}")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")
print(f"Unique entities: {df['Entity'].nunique()}")

After year filter (≥ 2000): (2206, 22)
Year range: 2000 – 2023
Unique entities: 116


#### 3. Drop Non-Sovereign Nations

In [14]:
def is_sovereign(iso3_code):
    """Check if an ISO3 code belongs to a sovereign nation."""
    if pd.isna(iso3_code) or not isinstance(iso3_code, str):
        return False
    try:
        country = pycountry.countries.get(alpha_3=iso3_code)
        return country is not None
    except Exception:
        return False

# Identify valid vs invalid entities
entities = df[['Entity', 'Code']].drop_duplicates()
entities['is_sovereign'] = entities['Code'].apply(is_sovereign)

valid_entities = entities[entities['is_sovereign']]
invalid_entities = entities[~entities['is_sovereign']]

print(f" Valid sovereign nations: {len(valid_entities)}")
print(f" Dropped (non-sovereign / invalid): {len(invalid_entities)}")

if len(invalid_entities) > 0:
    print("\nDropped entities:")
    for _, row in invalid_entities.iterrows():
        print(f"   {row['Code']} — {row['Entity']}")

 Valid sovereign nations: 116
 Dropped (non-sovereign / invalid): 0


In [15]:
# Keep only sovereign nations
valid_codes = valid_entities['Code'].tolist()
df = df[df['Code'].isin(valid_codes)].copy()
print(f"After sovereign filter: {df.shape}")
print(f"Countries: {df['Entity'].nunique()}")

After sovereign filter: (2206, 22)
Countries: 116


#### 4. Inspect & Handle Missing Values

In [16]:
print("--- Missing values by column ---")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")
print(f"Total cells: {df.shape[0] * df.shape[1]}")
pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
print(f"Missing %: {pct:.2f}%")

--- Missing values by column ---
5-9 years      20
10-14 years    20
15-19 years    20
20-24 years    20
25-29 years    20
30-34 years    20
35-39 years    20
40-44 years    20
45-49 years    20
50-54 years    20
55-59 years    20
60-64 years    20
65-69 years    20
70-74 years    20
75-79 years    20
80-84 years    38
Over 85s       38
dtype: int64

Total missing cells: 376
Total cells: 48532
Missing %: 0.77%


In [17]:
# Show which countries have missing values
age_cols = [c for c in df.columns if c not in ['Entity', 'Code', 'Year']]
missing_countries = df[df[age_cols].isnull().any(axis=1)][['Entity', 'Year']].drop_duplicates()
print(f"Countries with missing age-group data: {missing_countries['Entity'].nunique()}")
print()
for country in sorted(missing_countries['Entity'].unique()):
    years = sorted(missing_countries[missing_countries['Entity'] == country]['Year'].tolist())
    print(f"  {country}: years {years}")

Countries with missing age-group data: 8

  Bahrain: years [2000, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009]
  Cyprus: years [2000]
  Egypt: years [2012, 2013]
  Iraq: years [2015]
  Jordan: years [2015]
  Qatar: years [2020, 2021]
  Seychelles: years [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2019, 2020, 2021]
  Sri Lanka: years [2015, 2019]


In [18]:
# Strategy: Forward-fill within each country, then back-fill, then interpolate
# This preserves real trends while filling gaps

df = df.sort_values(['Entity', 'Year']).reset_index(drop=True)

# Interpolate within each country group
df[age_cols] = df.groupby('Entity')[age_cols].transform(
    lambda grp: grp.interpolate(method='linear', limit_direction='both')
)

# Any remaining NaN (e.g., country with all NaN in a column) → fill with column median
remaining_nulls = df[age_cols].isnull().sum().sum()
if remaining_nulls > 0:
    print(f"Remaining nulls after interpolation: {remaining_nulls}")
    for col in age_cols:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col]=df[col].fillna(median_val)
            print(f"  Filled {col} NaNs with median: {median_val:.2f}")

print(f"\n Missing values after cleaning: {df.isnull().sum().sum()}")

Remaining nulls after interpolation: 340
  Filled 5-9 years NaNs with median: 0.59
  Filled 10-14 years NaNs with median: 0.96
  Filled 15-19 years NaNs with median: 2.35
  Filled 20-24 years NaNs with median: 4.03
  Filled 25-29 years NaNs with median: 6.58
  Filled 30-34 years NaNs with median: 11.17
  Filled 35-39 years NaNs with median: 19.89
  Filled 40-44 years NaNs with median: 37.62
  Filled 45-49 years NaNs with median: 68.19
  Filled 50-54 years NaNs with median: 118.97
  Filled 55-59 years NaNs with median: 195.83
  Filled 60-64 years NaNs with median: 326.96
  Filled 65-69 years NaNs with median: 540.00
  Filled 70-74 years NaNs with median: 921.67
  Filled 75-79 years NaNs with median: 1631.06
  Filled 80-84 years NaNs with median: 2897.91
  Filled Over 85s NaNs with median: 6817.80

 Missing values after cleaning: 0


#### 5. Rename Columns for Consistency

In [19]:
# Rename to cleaner format
rename_map = {
    'Entity': 'Country',
    'Code': 'ISO3',
}
df.rename(columns=rename_map, inplace=True)

print(f"Final columns: {df.columns.tolist()}")
print(f"Final shape: {df.shape}")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")
print(f"Countries: {df['Country'].nunique()}")
df.head()

Final columns: ['Country', 'ISO3', 'Year', 'Under 1s', '1-4 years', '5-9 years', '10-14 years', '15-19 years', '20-24 years', '25-29 years', '30-34 years', '35-39 years', '40-44 years', '45-49 years', '50-54 years', '55-59 years', '60-64 years', '65-69 years', '70-74 years', '75-79 years', '80-84 years', 'Over 85s']
Final shape: (2206, 22)
Year range: 2000 – 2023
Countries: 116


,Country,ISO3,Year,Under 1s,1-4 years,5-9 years,10-14 years,15-19 years,20-24 years,25-29 years,...,40-44 years,45-49 years,50-54 years,55-59 years,60-64 years,65-69 years,70-74 years,75-79 years,80-84 years,Over 85s
0,Albania,ALB,2000,58.407078,17.234776,7.144820,4.170390,8.088235,10.349289,12.296882,...,41.291687,83.218704,134.50293,232.17247,517.48970,1013.75520,2187.6138,4181.5386,5780.9520,16228.261
1,Albania,ALB,2001,120.949165,22.952837,8.891436,4.611851,9.482332,13.786448,13.614449,...,39.514300,79.488930,123.96493,221.41757,493.51544,900.62320,1872.9281,3294.2620,5387.2190,8897.083
2,Albania,ALB,2002,93.905710,30.028480,7.706974,5.216452,7.615062,12.658014,15.913803,...,45.039660,63.288994,144.12244,194.17476,464.93982,958.47700,1798.0347,3681.9556,5869.7500,11045.224
3,Albania,ALB,2003,124.871160,35.511536,8.935659,5.300706,8.439339,14.867618,15.544042,...,53.754456,70.783646,147.65398,244.86769,456.18670,952.40110,2065.1438,3709.5312,6441.7640,13442.277
4,Albania,ALB,2004,119.201760,25.046040,11.081798,7.116333,5.723152,7.938788,11.926113,...,43.205680,77.464860,136.76787,233.49783,474.52652,905.28046,1929.5828,3564.4170,6252.4917,12437.811


#### 6. Quick Sanity Check

In [20]:
# Summary statistics for the age-group death rates
age_cols_final = [c for c in df.columns if c not in ['Country', 'ISO3', 'Year']]
print("--- Death Rate Summary Statistics (per 100k) ---")
print(df[age_cols_final].describe().round(1).to_string())
print()
print(f"Sample countries: {sorted(df['Country'].unique())[:10]}")

--- Death Rate Summary Statistics (per 100k) ---
       Under 1s  1-4 years  5-9 years  10-14 years  15-19 years  20-24 years  25-29 years  30-34 years  35-39 years  40-44 years  45-49 years  50-54 years  55-59 years  60-64 years  65-69 years  70-74 years  75-79 years  80-84 years  Over 85s
count    2206.0     2206.0     2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0       2206.0    2206.0
mean       16.2        2.3        1.2          1.7          3.4          5.6          9.0         15.5         27.3         51.0         91.7        158.8        262.6        436.6        715.9       1208.9       2089.2       3653.1    8073.6
std        39.1        4.5        2.2          3.0          3.8          5.7          8.4         13.6         22.3         38.9         68.3        117.1        192.7        325.2        522.3        858.4       1412.0   

## 7. Save Cleaned Dataset

In [ ]:
OUTPUT_PATH = '../data/preprocessed/age_group_mortality.csv'

df.to_csv(OUTPUT_PATH, index=False)

print(f" Saved cleaned dataset to: {OUTPUT_PATH}")
print(f"   Shape: {df.shape}")
print(f"   Countries: {df['Country'].nunique()}")
print(f"   Years: {df['Year'].min()} – {df['Year'].max()}")
print(f"   Age groups: {len(age_cols_final)}")
print()
print("Done! ")

 Saved cleaned dataset to: ../data/preprocessed/age_group_mortality.csv
   Shape: (2206, 22)
   Countries: 116
   Years: 2000 – 2023
   Age groups: 19

Done! 


: 